In [1]:
import pandas as pd
from collections import defaultdict

def load_initial_scores(csv_path, node_to_index):
    """
    Loads ranking scores from a CSV file.
    Robustly handles variations in column names (e.g. "Node ID"/"Order" vs "node"/"rank").
    """
    df = pd.read_csv(csv_path)
    
    # 1. Normalize column names (strip whitespace)
    df.columns = [str(c).strip() for c in df.columns]
    
    # 2. Detect Node Column
    node_col = None
    # Priority: "Node ID", then "node_id", then "node", then "id"
    for candidate in ["Node ID", "node_id", "node", "id", "Node"]:
        if candidate in df.columns:
            node_col = candidate
            break
            
    # 3. Detect Rank Column
    rank_col = None
    # Priority: "Order", then "rank", then "score"
    for candidate in ["Order", "rank", "score", "Rank", "order"]:
        if candidate in df.columns:
            rank_col = candidate
            break
    
    # 4. Fallback if specific headers aren't found but we have at least 2 columns
    if (node_col is None or rank_col is None) and len(df.columns) >= 2:
        node_col = df.columns[0]
        rank_col = df.columns[1]
    
    if node_col is None or rank_col is None:
        raise ValueError(f"Could not identify Node/Rank columns in {csv_path}. Columns found: {list(df.columns)}")

    # 5. Build the map
    # Ensure node IDs are treated as strings to match node_to_index keys
    df[node_col] = df[node_col].astype(str).str.strip()
    
    # Create dictionary: Node(str) -> Rank(int)
    # coerce errors='coerce' to handle non-numeric headers if they exist in data rows
    rank_values = pd.to_numeric(df[rank_col], errors='coerce').fillna(0).astype(int)
    rank_map = dict(zip(df[node_col], rank_values))

    # 6. Map to internal indices
    scores = {}
    for node_str, idx in node_to_index.items():
        # Ensure the lookup key matches the type in rank_map (string)
        s_node = str(node_str)
        if s_node in rank_map:
            scores[idx] = rank_map[s_node]

    # 7. Handle missing nodes (assign new ranks at the end)
    if scores:
        max_rank = max(scores.values()) + 1
    else:
        max_rank = 1
        
    for node_str, idx in node_to_index.items():
        if idx not in scores:
            scores[idx] = max_rank
            max_rank += 1

    return scores

def read_graph(file_path):
    """
    Parses a DIMACS graph file (extension .d or .gr).
    Lines start with 'a' for arcs: a <source> <target> <weight>
    """
    edges = []
    
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Skip comments ('c') and problem definition ('p')
            if line.startswith('c') or line.startswith('p'):
                continue
            
            # Parse arc lines: a <source> <target> <weight>
            if line.startswith('a'):
                parts = line.split()
                # parts[0] is 'a'
                u = parts[1]
                v = parts[2]
                
                # Handle weight if present, else default to 1.0
                if len(parts) > 3:
                    w = float(parts[3])
                else:
                    w = 1.0
                
                edges.append((u, v, w))

    # Create a sorted set of unique nodes to establish a deterministic index mapping
    node_set = sorted(set(u for u, v, _ in edges).union(v for u, v, _ in edges))
    
    # Map node IDs (strings) to 0-based integers
    node_to_index = {node: i for i, node in enumerate(node_set)}
    index_to_node = {i: node for node, i in node_to_index.items()}
    
    # Rebuild edges using the internal 0-based indices
    edges_indexed = [(node_to_index[u], node_to_index[v], w) for (u, v, w) in edges]
    
    return edges_indexed, node_to_index, index_to_node

from collections import Counter, defaultdict

def check_parallel_edges(edges_indexed, index_to_node=None, show_top=10):
    """
    Detect parallel directed edges: multiple entries with the same (u,v).
    edges_indexed: list of (u, v, w) with u,v as ints
    index_to_node: optional dict[int -> original node id] for nicer printing
    """
    pair_counts = Counter((u, v) for (u, v, w) in edges_indexed)

    parallel_pairs = [(pair, cnt) for pair, cnt in pair_counts.items() if cnt > 1]
    parallel_pairs.sort(key=lambda x: x[1], reverse=True)

    if not parallel_pairs:
        print("✅ No parallel edges found: every (u,v) pair appears exactly once.")
        return False

    total_edges = len(edges_indexed)
    num_unique_pairs = len(pair_counts)
    num_parallel_pairs = len(parallel_pairs)
    extra_edges_due_to_parallelism = total_edges - num_unique_pairs

    print("⚠️ Parallel edges FOUND.")
    print(f"  Total edges (rows):            {total_edges}")
    print(f"  Unique (u,v) pairs:            {num_unique_pairs}")
    print(f"  Pairs with multiplicity > 1:   {num_parallel_pairs}")
    print(f"  Extra edges beyond uniques:    {extra_edges_due_to_parallelism}")

    # Also show weight stats for top repeated pairs
    weights_by_pair = defaultdict(list)
    for u, v, w in edges_indexed:
        weights_by_pair[(u, v)].append(w)

    print(f"\nTop {min(show_top, len(parallel_pairs))} repeated (u,v) pairs:")
    for (u, v), cnt in parallel_pairs[:show_top]:
        ws = weights_by_pair[(u, v)]
        w_sum = sum(ws)
        w_max = max(ws)
        w_min = min(ws)
        if index_to_node is not None:
            u0 = index_to_node.get(u, u)
            v0 = index_to_node.get(v, v)
            print(f"  ({u}->{v})  aka ({u0}->{v0})  count={cnt}  sumW={w_sum:.6g}  minW={w_min:.6g}  maxW={w_max:.6g}")
        else:
            print(f"  ({u}->{v})  count={cnt}  sumW={w_sum:.6g}  minW={w_min:.6g}  maxW={w_max:.6g}")

    return True

# ---- Example usage ----
# edges_indexed, node_to_index, index_to_node = read_graph("/path/to/graph.gr")
# has_parallel = check_parallel_edges(edges_indexed, index_to_node=index_to_node, show_top=10)


# ==========================================
#               MAIN EXECUTION
# ==========================================

# === INPUTS ===
import os

# === INPUTS ===
from collections import defaultdict, Counter

edge_file = "connectome.d"
#ranking_file = f"{edge_file.replace('.d', '')}_lns_best_incumbent_ranking.csv"
#ranking_file="connectome_dimacs_hybrid_ranking_learning-48cpu.csv"
ranking_file="connectome_rocketcrane_ranking_20260206-161126.csv"

print(f"--- Processing {edge_file} ---")

# === 1. LOAD GRAPH (Using DIMACS Parser) ===
try:
    edges_indexed, node_to_index, index_to_node = read_graph(edge_file)
    total_weight = sum(w for _, _, w in edges_indexed)
    print(f"✅ Graph loaded: {len(node_to_index)} nodes, {len(edges_indexed)} edges.")
except Exception as e:
    print(f"❌ Error reading graph file: {e}")
    raise

# === 1b. CHECK PARALLEL EDGES (same (u,v) appears multiple times) ===
pair_counts = Counter((u, v) for (u, v, w) in edges_indexed)
parallel_pairs = [(pair, cnt) for pair, cnt in pair_counts.items() if cnt > 1]
parallel_pairs.sort(key=lambda x: x[1], reverse=True)

if not parallel_pairs:
    print("✅ No parallel edges: every directed pair (u,v) appears exactly once.")
else:
    total_edges = len(edges_indexed)
    unique_pairs = len(pair_counts)
    extra_edges = total_edges - unique_pairs
    print("⚠️ Parallel edges FOUND.")
    print(f"  Unique (u,v) pairs:         {unique_pairs}")
    print(f"  Pairs with multiplicity>1:  {len(parallel_pairs)}")
    print(f"  Extra edges beyond uniques: {extra_edges}")

    # show a few examples + weight aggregation
    weights_by_pair = defaultdict(list)
    for u, v, w in edges_indexed:
        weights_by_pair[(u, v)].append(w)

    print("\n  Top repeated pairs (showing up to 10):")
    for (u, v), cnt in parallel_pairs[:10]:
        ws = weights_by_pair[(u, v)]
        u_name = index_to_node[u]
        v_name = index_to_node[v]
        print(f"    ({u_name}->{v_name}) count={cnt}  sumW={sum(ws):.6g}  minW={min(ws):.6g}  maxW={max(ws):.6g}")

# === 2. LOAD RANKING (Using Robust Loader) ===
try:
    # load_initial_scores returns a dict: {internal_index: rank}
    scores = load_initial_scores(ranking_file, node_to_index)
    print(f"✅ Ranking loaded for {len(scores)} nodes.")
except Exception as e:
    print(f"❌ Error reading ranking file: {e}")
    raise

# Optional: sanity check all nodes present
if len(scores) != len(node_to_index):
    missing = set(node_to_index.values()) - set(scores.keys())
    print(f"⚠️ Ranking missing {len(missing)} nodes (showing up to 10): {list(missing)[:10]}")

# === 3. COMPUTE STATISTICS ===
forward_weight = 0.0
sample_debug = []

for u_idx, v_idx, w in edges_indexed:
    rank_u = scores[u_idx]
    rank_v = scores[v_idx]

    is_forward = rank_u < rank_v
    if is_forward:
        forward_weight += w

    if len(sample_debug) < 10:
        u_name = index_to_node[u_idx]
        v_name = index_to_node[v_idx]
        sample_debug.append((u_name, rank_u, v_name, rank_v, is_forward, w))

# === 4. PRINT RESULTS ===
backward_weight = total_weight - forward_weight

print(f"\n📊 FINAL RESULTS")
print(f"Total Edge Weight:        {total_weight:.2f}")
print(f"✅ Total Forward Weight:  {forward_weight:.2f}")
print(f"❌ Total Backward Weight: {backward_weight:.2f}")
print(f"📈 Forward Edge Ratio:    {forward_weight / total_weight:.6f}")

# === 5. DUPLICATE CHECK (ranks must be unique) ===
rank_to_nodes = defaultdict(list)
for idx, rank in scores.items():
    node_name = index_to_node[idx]
    rank_to_nodes[rank].append(node_name)

duplicates = {rank: nodes for rank, nodes in rank_to_nodes.items() if len(nodes) > 1}

if duplicates:
    print(f"\n❌ Detected {len(duplicates)} duplicate ranks.")
    for rank, nodes in sorted(duplicates.items())[:10]:
        print(f"  Rank {rank} assigned to {len(nodes)} nodes: {nodes[:5]}...")
else:
    print("\n✅ All ranks are unique.")


--- Processing connectome.d ---
✅ Graph loaded: 136648 nodes, 5657719 edges.
✅ No parallel edges: every directed pair (u,v) appears exactly once.
✅ Ranking loaded for 136648 nodes.

📊 FINAL RESULTS
Total Edge Weight:        41912141.00
✅ Total Forward Weight:  33428713.00
❌ Total Backward Weight: 8483428.00
📈 Forward Edge Ratio:    0.797590

✅ All ranks are unique.


In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import itertools
import numpy as np
import pandas as pd


# ============================================================
#                    I/O HELPERS
# ============================================================

def load_initial_scores(csv_path, node_to_index):
    """
    Loads ranking scores from a CSV file.
    Robustly handles variations in column names (e.g. "Node ID"/"Order" vs "node"/"rank").
    Returns: dict {internal_idx(int) -> score(int)}
    """
    df = pd.read_csv(csv_path)

    # Normalize column names
    df.columns = [str(c).strip() for c in df.columns]

    # Detect node column
    node_col = None
    for candidate in ["Node ID", "node_id", "node", "id", "Node"]:
        if candidate in df.columns:
            node_col = candidate
            break

    # Detect rank/score column
    rank_col = None
    for candidate in ["Order", "rank", "score", "Rank", "order"]:
        if candidate in df.columns:
            rank_col = candidate
            break

    # Fallback: first two columns
    if (node_col is None or rank_col is None) and len(df.columns) >= 2:
        node_col = df.columns[0]
        rank_col = df.columns[1]

    if node_col is None or rank_col is None:
        raise ValueError(
            f"Could not identify Node/Rank columns in {csv_path}. Columns found: {list(df.columns)}"
        )

    # Build node->rank map (strings)
    df[node_col] = df[node_col].astype(str).str.strip()
    rank_values = pd.to_numeric(df[rank_col], errors="coerce").fillna(0).astype(int)
    rank_map = dict(zip(df[node_col], rank_values))

    # Map to internal indices
    scores = {}
    for node_str, idx in node_to_index.items():
        s_node = str(node_str)
        if s_node in rank_map:
            scores[idx] = int(rank_map[s_node])

    # Fill missing nodes with ranks at the end
    max_rank = (max(scores.values()) + 1) if scores else 1
    for node_str, idx in node_to_index.items():
        if idx not in scores:
            scores[idx] = max_rank
            max_rank += 1

    return scores


def read_graph(file_path):
    """
    Parses a DIMACS graph file (.d or .gr).
    Arc lines: a <source> <target> <weight>
    Returns:
      edges_indexed: list[(u_idx, v_idx, w)]
      node_to_index: dict node_str->idx
      index_to_node: dict idx->node_str
    """
    edges = []
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("c") or line.startswith("p"):
                continue
            if line.startswith("a"):
                parts = line.split()
                u = parts[1]
                v = parts[2]
                w = float(parts[3]) if len(parts) > 3 else 1.0
                edges.append((u, v, w))

    node_set = sorted(set(u for u, v, _ in edges).union(v for u, v, _ in edges))
    node_to_index = {node: i for i, node in enumerate(node_set)}
    index_to_node = {i: node for node, i in node_to_index.items()}
    edges_indexed = [(node_to_index[u], node_to_index[v], w) for (u, v, w) in edges]
    return edges_indexed, node_to_index, index_to_node


# ============================================================
#               RANKINGS -> POSITIONS
# ============================================================

def scores_to_positions(scores_dict, n_nodes):
    """
    Convert {idx -> score} into strict positions 1..n_nodes.
    Lower score = earlier position. Ties broken by node index.
    """
    items = [(idx, scores_dict[idx]) for idx in range(n_nodes)]
    items.sort(key=lambda x: (x[1], x[0]))
    pos = np.empty(n_nodes, dtype=np.int32)
    for p, (idx, _) in enumerate(items, start=1):
        pos[idx] = p
    return pos


# ============================================================
#             DISTANCE / SIMILARITY METRICS
# ============================================================

def summarize_differences(diffs):
    """
    diffs: signed position differences, length n
    """
    diffs = np.asarray(diffs, dtype=np.int64)
    absdiff = np.abs(diffs)

    return {
        "n": int(diffs.size),

        "mean_abs": float(absdiff.mean()),
        "median_abs": float(np.median(absdiff)),
        "p90_abs": float(np.percentile(absdiff, 90)),
        "p95_abs": float(np.percentile(absdiff, 95)),
        "p99_abs": float(np.percentile(absdiff, 99)),
        "max_abs": int(absdiff.max()),

        "mean_signed": float(diffs.mean()),
        "std_signed": float(diffs.std(ddof=0)),
        "min_signed": int(diffs.min()),
        "max_signed": int(diffs.max()),
    }


def topk_overlap(posA, posB, k):
    """
    Fractional overlap of top-k sets (smallest positions).
    Returns: overlap_size / k
    """
    n = posA.size
    k = int(min(max(1, k), n))
    topA = set(np.argpartition(posA, k - 1)[:k].tolist())
    topB = set(np.argpartition(posB, k - 1)[:k].tolist())
    return len(topA & topB) / float(k)


def bucket_overlap_profile(posA, posB, buckets=(100, 500, 1000, 5000, 10000)):
    """
    Compute overlap fractions for multiple top-k buckets.
    Returns dict with keys like overlap_top_100, overlap_top_500, ...
    """
    out = {}
    for k in buckets:
        out[f"overlap_top_{k}"] = float(topk_overlap(posA, posB, k))
    return out


def spearman_rho_from_positions(posA, posB):
    """
    Spearman rho for strict permutations:
      rho = 1 - 6 * sum(d_i^2) / (n(n^2-1))
    where d_i = posA[i] - posB[i].
    """
    n = posA.size
    d = (posA.astype(np.int64) - posB.astype(np.int64))
    ssd = float(np.dot(d, d))
    denom = n * (n * n - 1)
    if denom == 0:
        return 0.0
    return float(1.0 - (6.0 * ssd) / denom)


def approx_kendall_tau_b(posA, posB, n_samples=200000, seed=1):
    """
    Approximate Kendall tau via random pair sampling (fast, scalable).
    For permutations, tau ≈ (concordant - discordant) / total_pairs_sampled

    This is NOT exact Kendall (O(n log n) possible), but enough to support/attack claims.
    """
    rng = np.random.default_rng(seed)
    n = posA.size
    if n < 2:
        return 0.0

    # sample pairs (i,j)
    i = rng.integers(0, n, size=n_samples, dtype=np.int32)
    j = rng.integers(0, n, size=n_samples, dtype=np.int32)
    mask = (i != j)
    i = i[mask]
    j = j[mask]

    # concordant if order matches in both rankings
    a = posA[i] < posA[j]
    b = posB[i] < posB[j]
    conc = np.sum(a == b)
    disc = (a.size - conc)
    tot = a.size
    if tot == 0:
        return 0.0
    return float((conc - disc) / tot)


# ============================================================
#             QUALITY METRICS (FORWARD WEIGHT)
# ============================================================

def forward_backward_weight_fast(edges_indexed, positions):
    """
    Forward edge: positions[u] < positions[v] for edge u->v.
    Returns: (forward_w, backward_w, total_w, forward_ratio)
    """
    U = np.fromiter((e[0] for e in edges_indexed), dtype=np.int32)
    V = np.fromiter((e[1] for e in edges_indexed), dtype=np.int32)
    W = np.fromiter((e[2] for e in edges_indexed), dtype=np.float64)

    total = float(W.sum())
    mask = positions[U] < positions[V]
    fw = float(W[mask].sum())
    bw = total - fw
    fr = (fw / total) if total > 0 else 0.0
    return fw, bw, total, fr


def quality_gap_metrics(infoA, infoB):
    """
    Useful for claim-testing:
    - absolute and relative forward weight gaps
    - ratio gap
    """
    fwA, fwB = infoA["forward_w"], infoB["forward_w"]
    frA, frB = infoA["forward_ratio"], infoB["forward_ratio"]
    total = infoA["total_w"]  # same graph so should match
    return {
        "forward_w_gap": float(abs(fwA - fwB)),
        "forward_ratio_gap": float(abs(frA - frB)),
        "forward_w_gap_frac_total": float(abs(fwA - fwB) / total) if total > 0 else 0.0,
    }


# ============================================================
#        MAIN DRIVER: CLAIM-ORIENTED REPORTING
# ============================================================

def analyze_rankings_for_graph(
    graph_path,
    ranking_csv_paths,
    out_dir="ranking_similarity_outputs",
    deduplicate_inputs=True,

    # claim tests / extras
    compute_pairwise=True,
    compute_reference_distances=True,   # distances to best ranking
    compute_topk_overlaps=True,
    topk_buckets=(100, 500, 1000, 5000, 10000),

    compute_spearman=True,
    compute_kendall_approx=True,
    kendall_samples=200000,
    kendall_seed=1,

    # cluster helpers
    build_family_summary=True,          # summarize “clusters” by distance threshold
    family_threshold_mean_abs=2000,     # “close” if mean_abs <= threshold
):
    """
    Produces multiple CSVs to prove/disprove claims like:
      - "Higher forward_w implies convergence"
      - "Good solutions cluster"
      - "Same method is stable across CPU counts"
      - "Near-optimal rankings can still be far apart"
      - "Top-k nodes overlap a lot / little"
      - "Rank correlations are high / low"

    Outputs (in out_dir):
      1) ranking_quality.csv
      2) pairwise_metrics.csv
      3) reference_to_best.csv (optional)
      4) family_summary.csv (optional)

    Returns:
      ranking_quality_df, pairwise_df, reference_df, family_df
    """
    edges_indexed, node_to_index, index_to_node = read_graph(graph_path)
    n = len(node_to_index)

    # Deduplicate input paths
    csv_paths = list(ranking_csv_paths)
    if deduplicate_inputs:
        seen = set()
        deduped = []
        for p in csv_paths:
            key = os.path.abspath(p)
            if key not in seen:
                seen.add(key)
                deduped.append(p)
        csv_paths = deduped

    os.makedirs(out_dir, exist_ok=True)

    # Load all rankings
    ranking_info = {}  # name -> {pos, forward_w, ...}
    for csv_path in csv_paths:
        name = os.path.splitext(os.path.basename(csv_path))[0]
        if name in ranking_info:
            # ensure unique
            suffix = 2
            new_name = f"{name}__{suffix}"
            while new_name in ranking_info:
                suffix += 1
                new_name = f"{name}__{suffix}"
            name = new_name

        scores = load_initial_scores(csv_path, node_to_index)
        pos = scores_to_positions(scores, n)
        fw, bw, total, fr = forward_backward_weight_fast(edges_indexed, pos)

        ranking_info[name] = {
            "ranking": name,
            "csv_path": csv_path,
            "pos": pos,
            "forward_w": fw,
            "backward_w": bw,
            "total_w": total,
            "forward_ratio": fr,
        }

    # 1) Per-ranking quality
    ranking_quality_df = pd.DataFrame([
        {
            "ranking": info["ranking"],
            "csv_path": info["csv_path"],
            "n_nodes": n,
            "forward_w": info["forward_w"],
            "backward_w": info["backward_w"],
            "total_w": info["total_w"],
            "forward_ratio": info["forward_ratio"],
        }
        for info in ranking_info.values()
    ]).sort_values(by=["forward_w", "forward_ratio"], ascending=False)

    ranking_quality_df.to_csv(os.path.join(out_dir, "ranking_quality.csv"), index=False)

    # Identify best ranking by forward_w
    best_name = str(ranking_quality_df.iloc[0]["ranking"])
    best_pos = ranking_info[best_name]["pos"]

    # 2) Pairwise metrics
    pairwise_df = None
    if compute_pairwise:
        rows = []
        items = list(ranking_info.items())

        for (nameA, infoA), (nameB, infoB) in itertools.combinations(items, 2):
            posA, posB = infoA["pos"], infoB["pos"]
            diffs = posA.astype(np.int64) - posB.astype(np.int64)

            base = {
                "ranking_A": nameA,
                "ranking_B": nameB,

                "forward_w_A": infoA["forward_w"],
                "forward_ratio_A": infoA["forward_ratio"],
                "forward_w_B": infoB["forward_w"],
                "forward_ratio_B": infoB["forward_ratio"],
            }
            base.update(quality_gap_metrics(infoA, infoB))
            base.update(summarize_differences(diffs))

            # normalized distances (very useful for claims)
            base["mean_abs_frac_n"] = float(base["mean_abs"] / n)
            base["p95_abs_frac_n"] = float(base["p95_abs"] / n)
            base["max_abs_frac_n"] = float(base["max_abs"] / n)

            if compute_spearman:
                base["spearman_rho"] = spearman_rho_from_positions(posA, posB)

            if compute_kendall_approx:
                base["kendall_tau_approx"] = approx_kendall_tau_b(
                    posA, posB, n_samples=kendall_samples, seed=kendall_seed
                )

            if compute_topk_overlaps:
                base.update(bucket_overlap_profile(posA, posB, buckets=topk_buckets))

            rows.append(base)

        pairwise_df = pd.DataFrame(rows).sort_values(
            by=["mean_abs", "p95_abs", "max_abs"], ascending=True
        )
        pairwise_df.to_csv(os.path.join(out_dir, "pairwise_metrics.csv"), index=False)

    # 3) Distances to best ranking (tests "convergence to best")
    reference_df = None
    if compute_reference_distances:
        rows = []
        best_info = ranking_info[best_name]
        for name, info in ranking_info.items():
            if name == best_name:
                continue
            diffs = best_pos.astype(np.int64) - info["pos"].astype(np.int64)
            stats = summarize_differences(diffs)
            row = {
                "best_ranking": best_name,
                "other_ranking": name,

                "best_forward_w": best_info["forward_w"],
                "best_forward_ratio": best_info["forward_ratio"],

                "other_forward_w": info["forward_w"],
                "other_forward_ratio": info["forward_ratio"],

                "forward_w_gap": abs(best_info["forward_w"] - info["forward_w"]),
                "forward_ratio_gap": abs(best_info["forward_ratio"] - info["forward_ratio"]),
            }
            row.update(stats)
            row["mean_abs_frac_n"] = float(row["mean_abs"] / n)
            row["p95_abs_frac_n"] = float(row["p95_abs"] / n)

            if compute_spearman:
                row["spearman_rho"] = spearman_rho_from_positions(best_pos, info["pos"])
            if compute_topk_overlaps:
                row.update(bucket_overlap_profile(best_pos, info["pos"], buckets=topk_buckets))

            rows.append(row)

        reference_df = pd.DataFrame(rows).sort_values(
            by=["other_forward_w"], ascending=False
        )
        reference_df.to_csv(os.path.join(out_dir, "reference_to_best.csv"), index=False)

    # 4) Quick family/cluster summary (simple thresholding)
    family_df = None
    if build_family_summary and pairwise_df is not None:
        # Build adjacency if close by mean_abs
        names = list(ranking_info.keys())
        idx_map = {name: i for i, name in enumerate(names)}
        parent = list(range(len(names)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra

        # Union pairs with mean_abs <= threshold
        for _, r in pairwise_df.iterrows():
            if float(r["mean_abs"]) <= float(family_threshold_mean_abs):
                union(idx_map[r["ranking_A"]], idx_map[r["ranking_B"]])

        # Collect components
        comp = {}
        for name in names:
            root = find(idx_map[name])
            comp.setdefault(root, []).append(name)

        # Summarize each family with best forward_w inside it
        fam_rows = []
        for root, members in comp.items():
            sub = ranking_quality_df[ranking_quality_df["ranking"].isin(members)]
            sub_sorted = sub.sort_values(by=["forward_w"], ascending=False)
            best_member = sub_sorted.iloc[0]["ranking"]
            best_fw = float(sub_sorted.iloc[0]["forward_w"])
            best_fr = float(sub_sorted.iloc[0]["forward_ratio"])
            fam_rows.append({
                "family_id": int(root),
                "size": int(len(members)),
                "best_member": str(best_member),
                "best_forward_w": best_fw,
                "best_forward_ratio": best_fr,
                "members": ";".join(sorted(members)),
            })

        family_df = pd.DataFrame(fam_rows).sort_values(
            by=["best_forward_w", "size"], ascending=False
        )
        family_df.to_csv(os.path.join(out_dir, "family_summary.csv"), index=False)

    # Print a small console summary for convenience
    print("\n=== Ranking quality (sorted by forward_w) ===")
    print(ranking_quality_df.to_string(index=False))

    if pairwise_df is not None:
        print("\n=== Pairwise similarity (top 25 closest by mean_abs) ===")
        print(pairwise_df.head(25).to_string(index=False))

        # The key claim test:
        # pairs with tiny forward_ratio_gap but large mean_abs (evidence of non-unique near-optima)
        probe = pairwise_df.copy()
        probe["forward_ratio_gap"] = probe["forward_ratio_gap"].astype(float)
        probe["mean_abs"] = probe["mean_abs"].astype(float)
        near = probe[probe["forward_ratio_gap"] <= 1e-4].sort_values("mean_abs", ascending=False).head(10)
        if len(near) > 0:
            print("\n=== Pairs with tiny forward_ratio_gap (<=1e-4) but large mean_abs (top 10) ===")
            cols = ["ranking_A", "ranking_B", "forward_ratio_gap", "forward_w_gap", "mean_abs", "p95_abs", "spearman_rho"]
            cols = [c for c in cols if c in near.columns]
            print(near[cols].to_string(index=False))

    if reference_df is not None:
        print("\n=== Distances to best ranking (sorted by other_forward_w desc) ===")
        print(reference_df.head(25).to_string(index=False))

    if family_df is not None:
        print("\n=== Family summary (clusters by mean_abs threshold) ===")
        print(family_df.to_string(index=False))

    return ranking_quality_df, pairwise_df, reference_df, family_df


# ============================================================
#                     EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":
    graph_path = "connectome.d"

    ranking_csvs = [
        "35435948.csv",
        "connectome_paper_fas_ranking_20260202_123552.csv",
        "connectome_dimacs_hybrid_ranking_learning-48cpu.csv",
     
        "bader.csv",
        "connectome_adaptive_out_over_inplus1_learning_ranking.csv",
       
        "connectome_paper_fas_ranking.csv",
        "connectome_hybrid_ranking-48cpu.csv",
        "connectome_hybrid_ranking-24cpu.csv",
        "connectome_hybrid_ranking-12cpu.csv",
        "35452425.csv",
        "35452425_ranking-24cpu.csv",
        "35452425_ranking-12cpu.csv",
        "35452425_ranking-48cpu.csv",
        "35435948_ranking-12cpu.csv",
        "35435948_ranking-24cpu.csv",
        "connectome_dimacs_scc_init.csv",
        "connectome_dimacs_scc_init_learning.csv"
        
    ]

    analyze_rankings_for_graph(
        graph_path=graph_path,
        ranking_csv_paths=ranking_csvs,
        out_dir="ranking_similarity_outputs",

        # claim-oriented toggles
        compute_pairwise=True,
        compute_reference_distances=True,
        compute_topk_overlaps=True,
        topk_buckets=(100, 500, 1000, 5000, 10000),

        compute_spearman=True,
        compute_kendall_approx=True,   # approximate, scalable
        kendall_samples=200000,
        kendall_seed=1,

        build_family_summary=True,
        family_threshold_mean_abs=2000,  # adjust to define “family/cluster”
    )



=== Ranking quality (sorted by forward_w) ===
                                              ranking                                                  csv_path  n_nodes  forward_w  backward_w    total_w  forward_ratio
                      connectome_hybrid_ranking-12cpu                       connectome_hybrid_ranking-12cpu.csv   136648 35463823.0   6448318.0 41912141.0       0.846147
                      connectome_hybrid_ranking-48cpu                       connectome_hybrid_ranking-48cpu.csv   136648 35463423.0   6448718.0 41912141.0       0.846137
                      connectome_hybrid_ranking-24cpu                       connectome_hybrid_ranking-24cpu.csv   136648 35463349.0   6448792.0 41912141.0       0.846135
                                                bader                                                 bader.csv   136648 35459266.0   6452875.0 41912141.0       0.846038
                               35452425_ranking-24cpu                                35452425_ranking-2

In [15]:
import os

def sanitize_id(s: str) -> str:
    # Your DIMACS reader splits on whitespace, so node IDs must not contain whitespace.
    return "_".join(str(s).strip().split())

def tsv_to_dimacs_clickstream(
    input_path: str,
    keep_types=("link",),      # set to None to keep ALL types
    min_weight: int = 1,
    transit_time_value: int = 1,
    write_transit_time: bool = True,
    skip_self_loops: bool = True,
    encoding: str = "utf-8",
):
    """
    Convert Wikimedia clickstream TSV to DIMACS-like .d file.

    Expected TSV columns (tab-separated):
      prev<TAB>curr<TAB>type<TAB>n

    Output:
      <same base name>.d in same folder, with lines:
        a <prev> <curr> <weight> <transit_time>

    Returns: (out_path, kept_arcs, skipped_rows)
    """
    base, _ = os.path.splitext(input_path)
    out_path = base + ".d"

    kept = 0
    skipped = 0

    with open(input_path, "r", encoding=encoding, errors="replace") as fin, \
         open(out_path, "w", encoding="utf-8") as fout:

        fout.write("c Converted from Wikimedia clickstream TSV\n")
        fout.write(f"c keep_types={keep_types} min_weight={min_weight} skip_self_loops={skip_self_loops}\n")

        for raw in fin:
            line = raw.rstrip("\n")
            if not line:
                continue

            parts = line.split("\t")
            if len(parts) < 4:
                skipped += 1
                continue

            prev, curr, typ, n_str = parts[0], parts[1], parts[2], parts[3]

            if keep_types is not None and typ not in keep_types:
                skipped += 1
                continue

            try:
                w = int(n_str)
            except ValueError:
                skipped += 1
                continue

            if w < min_weight:
                skipped += 1
                continue

            u = sanitize_id(prev)
            v = sanitize_id(curr)

            if skip_self_loops and u == v:
                skipped += 1
                continue

            if write_transit_time:
                fout.write(f"a {u} {v} {w} {transit_time_value}\n")
            else:
                fout.write(f"a {u} {v} {w}\n")

            kept += 1

        fout.write(f"c kept_arcs={kept} skipped_rows={skipped}\n")

    return out_path, kept, skipped
input_path = "clickstream-arwiki-2026-01.tsv"   # change to your path
out_path, kept, skipped = tsv_to_dimacs_clickstream(
    input_path,
    keep_types=("link",),   # or None to keep all types
    min_weight=5,           # optional filter to reduce size/noise
)

out_path, kept, skipped


('clickstream-arwiki-2026-01.d', 403197, 608958)

In [31]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kendalltau

# NOTE: Ensure 'read_graph' and 'load_initial_scores' are defined in your notebook
# before running this block, as they were called in your original code.

def unweighted_kendall_table_5(
    graph_csv_path: str,
    score_csv_path1: str,
    score_csv_path2: str,
    score_csv_path3: str,
    score_csv_path4: str,
    score_csv_path5: str,
    score_labels: list[str] | None = None,
    normalize: bool = True,
    return_tau: bool = True,
):
    """
    Returns:
      - dist_df: 5x5 table of *unweighted* Kendall distance
      - tau_df (optional): Kendall tau
    
    Optimized to use Scipy.stats.kendalltau to avoid O(N^2) memory crash.
    """
    score_paths = [
        score_csv_path1,
        score_csv_path2,
        score_csv_path3,
        score_csv_path4,
        score_csv_path5,
    ]

    if score_labels is None:
        score_labels = [os.path.splitext(os.path.basename(p))[0] for p in score_paths]
    if len(score_labels) != 5:
        raise ValueError("score_labels must have length 5 (or be None).")

    # --- Load graph (for node universe / indexing) ---
    # Expects read_graph to return edges, node_to_index, index_to_node
    edges_indexed, node_to_index, index_to_node = read_graph(graph_csv_path)

    n = len(node_to_index)
    if n < 2:
        raise ValueError("Need at least two nodes to compute Kendall distance.")

    # --- Total Pairs Calculation (Math only, no list creation) ---
    total_pairs = (n * (n - 1)) // 2

    # --- Load rankings (dict: idx -> rank) ---
    rankings_dicts = [load_initial_scores(p, node_to_index) for p in score_paths]

    # Convert dicts to aligned lists [rank of node 0, rank of node 1, ...]
    # This is required for optimized calculation
    ranking_arrays = []
    for k, scores in enumerate(rankings_dicts):
        if len(scores) != n:
            raise ValueError(
                f"Ranking {k} has {len(scores)} scored nodes but graph has n={n}."
            )
        # Create a list where index i contains the rank of node i
        ranking_arrays.append([scores[i] for i in range(n)])

    # --- Compute pairwise matrices ---
    m = 5
    dist_mat = [[0.0] * m for _ in range(m)]
    tau_mat = [[0.0] * m for _ in range(m)] if return_tau else None

    for i in range(m):
        for j in range(i + 1, m):
            # Use Scipy's optimized C implementation (O(N log N))
            # correlation is the standard Kendall Tau-b
            tau_val, _ = kendalltau(ranking_arrays[i], ranking_arrays[j])
            
            # --- Convert Tau back to Distance ---
            # Formula: tau = 1 - 2 * (discordant_fraction)
            # Therefore: discordant_fraction = (1 - tau) / 2
            
            # Handle potential floating point imprecision or NaN
            if np.isnan(tau_val):
                tau_val = 0.0 # Or handle as appropriate for your data
            
            d_norm = (1.0 - tau_val) / 2.0
            
            if normalize:
                dist_val = d_norm
            else:
                dist_val = d_norm * total_pairs

            dist_mat[i][j] = dist_mat[j][i] = dist_val

            if return_tau:
                tau_mat[i][j] = tau_mat[j][i] = tau_val

    # --- Diagonals ---
    for i in range(m):
        dist_mat[i][i] = 0.0
        if return_tau:
            tau_mat[i][i] = 1.0

    # --- Build DataFrames ---
    dist_df = pd.DataFrame(dist_mat, index=score_labels, columns=score_labels)
    dist_df.index.name = "Ranking"
    dist_df.columns.name = "Ranking"

    if normalize:
        dist_df.attrs["title"] = (
            "Unweighted Kendall distance (normalized): "
            "each entry = fraction of discordant pairs. "
            "Range [0,1]."
        )
    else:
        dist_df.attrs["title"] = (
            "Unweighted Kendall distance (raw): "
            "each entry = # discordant unordered node pairs."
        )

    if return_tau:
        tau_df = pd.DataFrame(tau_mat, index=score_labels, columns=score_labels)
        tau_df.index.name = "Ranking"
        tau_df.columns.name = "Ranking"
        tau_df.attrs["title"] = (
            "Unweighted Kendall tau: Range [-1,1], +1=identical, -1=reverse."
        )
        return dist_df, tau_df

    return dist_df


def print_table_with_title(df: pd.DataFrame, decimals: int = 6):
    """Print a title above a DataFrame (if present), then print the DataFrame."""
    title = df.attrs.get("title", "")
    if title:
        print("\n" + title)
        print("-" * len(title))
    print(df.round(decimals))


# -----------------------
# Execution Block
# -----------------------
# Ensure you have your helper functions defined before running this!

dist_df, tau_df = unweighted_kendall_table_5(
    graph_csv_path="connectome_graph.csv",
    score_csv_path1="35435948.csv",
    score_csv_path2="35435948_ranking-48cpu.csv",
    score_csv_path3="35435948_ranking-48cpu-second72.csv",
    score_csv_path4="bader.csv",
    score_csv_path5="connectome_hybrid_ranking-12cpu.csv",
    score_labels=["A", "B", "C", "D", "E"],
)

print_table_with_title(dist_df, decimals=6)
print_table_with_title(tau_df, decimals=6)


ValueError: Need at least two nodes to compute Kendall distance.

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

# === INPUT FILE ===
input_file = "/content/drive/MyDrive/ibm02_ranking.csv"
output_file = "/content/drive/MyDrive/ibm02_ranking.csv"

# === LOAD DATA ===
df = pd.read_csv(input_file)
df.columns = [col.strip() for col in df.columns]
df['Node ID'] = df['Node ID'].astype(str).str.strip()

# === SORT AND ASSIGN UNIQUE RANKS ===
df = df.sort_values(by='Order').reset_index(drop=True)
df['Order'] = range(len(df))

# === SAVE TO NEW FILE ===
df.to_csv(output_file, index=False)

print(f"✅ Corrected ranking saved to: {output_file}")
